# Explore the synthetic break corpus

After running `uv run sba generate-corpus --count 500`, use this notebook to:
1. Sanity-check the corpus distribution across categories, currencies, severities
2. Browse individual cases and copy interesting IDs into `data/golden_set.jsonl`
3. Spot pairs of cases that look like real precedent matches → those become your eval set

In [ ]:
import json
from collections import Counter

import pandas as pd

from sba.config import settings
from sba.synthetic.schemas import HistoricalBreak

In [ ]:
# Load corpus
breaks = []
with settings.corpus_path.open() as f:
    for line in f:
        breaks.append(HistoricalBreak.model_validate_json(line))

print(f'Loaded {len(breaks)} breaks')
df = pd.DataFrame([b.model_dump() for b in breaks])
df.head()

In [ ]:
# Distribution checks
print('Category distribution:')
print(df['category'].value_counts())
print('\nCurrency distribution:')
print(df['currency'].value_counts())
print('\nSeverity distribution:')
print(df['severity'].value_counts())

In [ ]:
# Browse a single case in detail
sample = breaks[42]
print(f'Case: {sample.case_id}')
print(f'Trade ID: {sample.trade_id}')
print(f'Category: {sample.category}')
print(f'\nNarrative:\n{sample.analyst_narrative}')
print(f'\nResolution:\n{sample.resolution}')

## Building the golden set

For 30 hand-labelled query → expected-match pairs:
1. Pick a case that you'd consider a 'new' break
2. Search the corpus manually for the case that's the closest precedent
3. Append `{"query_break_id": ..., "expected_match_id": ..., "notes": ...}` to `data/golden_set.jsonl`

Aim for coverage across all 6 categories and a mix of severities.